# 1. Importação de Dados para o BigQuery
Dataset: Mercado de Games

Este notebook é dedicado exclusivamente para pegar os arquivos CSV brutos (da pasta `data/` ou do Google Drive) e carregá-los como tabelas no seu dataset do Google BigQuery.

## 1. Módulos e Setup

In [ ]:
import pandas as pd
import os
from google.cloud import bigquery

# Opcional: Se rodar via Google Colab, descomente as linhas abaixo para montar o Drive e Autenticar
# from google.colab import auth, drive
# auth.authenticate_user()
# drive.mount('/content/drive')

## 2. Autenticação do Projeto

In [ ]:
# Substitua pelo seu Project ID real
project_id = 'seu_projeto'
dataset_id = 'seu_dataset'

client = bigquery.Client(project=project_id)

## 3. Listar Arquivos CSV (Exemplo Local ou Drive)

In [ ]:
# Caminho onde os CSVs estão salvos.
data_path = '../data/'
# Se for no Colab, ajuste para algo como: '/content/drive/MyDrive/sua_pasta/data/'

all_files = []
for root, dirs, files in os.walk(data_path):
    for file in files:
        if file.endswith('.csv'):
            all_files.append(os.path.join(root, file))

print(f"Encontrados {len(all_files)} arquivos.")
all_files

## 4. Subir Dados para o BigQuery

In [ ]:
for file_path in all_files:
    # Extrair o nome da tabela a partir do nome do arquivo (sem o .csv)
    file_name = os.path.basename(file_path)
    table_name = file_name.replace('.csv', '')
    
    # Definir id completo da tabela no BQ
    table_id = f"{project_id}.{dataset_id}.{table_name}"
    
    print(f"Carregando {file_name} para {table_id}...")
    
    # Ler CSV
    df = pd.read_csv(file_path)
    
    # Configurar Job (Substituir/Truncar a tabela caso já exista)
    job_config = bigquery.LoadJobConfig(write_disposition="WRITE_TRUNCATE")
    
    # Enviar ao BigQuery
    job = client.load_table_from_dataframe(df, table_id, job_config=job_config)
    job.result()
    
    print(f"-> Sucesso! Tabela {table_name} atualizada no BQ.\n")